In [ ]:
#Importacion de las librerias necesarias
#Uso del codigo que fue entregado como base del proyecto
import pandas as pd
import requests
import numpy as np
import matplotlib.pyplot as plt
import time
from IPython.display import display
#Configuración para tablas en Jupyter con el fin de mantener una presentacion adecuada y facil legibilidad de los datos
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)
#URL de la API
api_url = 'https://api.yelp.com/v3/businesses/search'
#Credenciales Yelp
clientid = 'GWOCZh9-BmZxtdsAjr7Gug'
apikey = 'FHVvXoNmTXIl9DuxYis7AV5uLPujm9MLwrhgs5NgvCfaOxd3V6mxt6dQU8eEqYJiGxe816XATx7ufWjbMWqbV-2Uku1jxBJv8BGRC74NroLPl27PDQqs0tDixit-YHYx'
headers = {'Authorization': f'Bearer {apikey}'}
#Ciudades disponibles para el analisis de los datos
ciudades = {
    1: 'Miami',
    2: 'Denver',
    3: 'Boston',
    4: 'San Diego',
    5: 'Dallas',
    6: 'NYC',
    7: 'Seattle',
    8: 'Chicago',
    9: 'Houston',
    10: 'Phoenix'
}
#Primer ciclo while para que al salir de uno de los siguientes ciclos no se interrumpa o acabe el codigo sino que caundo el usuario lo solicite o requiera pueda volver
#Esto con la finalidad de que el usuario pueda explorar los datos de las ciudades por separado y pueda cambiar de ciudad a su comodidad
#El usuario se movera atraves del codigo ingresando los digitos disponibles, cualquier valor fuera de rango le solicitara al usuario que trate nuevamente
#Incluso con valores erroneos el codigo no se interrumpira sino que insistira en recibir una de las respuestas validas, para que el codigo no se rompa mientras el usuario esta usandolo
while True:
    print('Bienvenido, dentro de poco tendra acceso a datos recolectados de varios restaurantes\nPara comodidad estan dividos en ciudades')
    print('\nCiudades disponibles\n')
    for numero, ciudad in ciudades.items():
        print(f'{numero}. {ciudad}')
    print('0. Salir')
    try:
        opcion_ciudad = int(input('\nSeleccione una ciudad: '))
        if opcion_ciudad == 0:
            print('\nHasta luego')
            break
        if opcion_ciudad not in ciudades:
            print('\nCiudad inválida')
            continue
        ciudad = ciudades[opcion_ciudad]
    except ValueError:
        print('\nDebe ingresar un número')
        continue
    print(f'\nConsultando Yelp para {ciudad}...')
    time.sleep(1)
    #Consulta inicial de los datos a solicitud del usuario
    params = {
        'term': 'restaurants',
        'location': ciudad,
        'limit': 1
    }
    #Obtener informacion del api, parametros y encabezados para sufutro uso
    response = requests.get(
        api_url,
        params=params,
        headers=headers
    )
    #Se hara un conteo total de los datos disponibles para la ciudad seleccionada y se le explicara al usurio las limitaciones con los datos
    #Esto mismo con la idea de que apesar de que el usuario no tenga acceso a la mayor parte de los datos sin tendra distintas opciones y datos que lo ayudaran con lo que necesite
    #Se presentaran los datos con el limite de 50 datos que se permiten en 4 diferentes paginas a las que el usuario podra cambiar cuando lo desee
    data = response.json()
    total_restaurantes = data['total']
    limite_api = min(
        total_restaurantes,
        1000
    )
    max_pagina = int(
        np.ceil(limite_api / 50)
    )

    print(f'\nYelp registra aproximadamente '
        f'{total_restaurantes:,} restaurantes en {ciudad}.')
    if total_restaurantes > 1000:
        print('\nPor limitaciones de la API de Yelp '
            'solo pueden consultarse los primeros '
            '200 resultados.')
    print(f'\nRestaurantes disponibles para análisis: '
        f'200')
    print(f'Puede seleccionar páginas entre '
        f'1 y 4.\nCada pagina contara con datos de 50 restaurantes')
    volver_ciudades = False
    #Segundo ciclo while donde el usuario tendra la opcion de analizar los datos desde y hasta la pagina que desee
    #Se hace control de excepciones para que no falle el codigo ni afecte a la experiencia del usuario
    while not volver_ciudades:
        try:
            pagina = int(input('\nSeleccione una página: '))
            if not (1 <= pagina <= max_pagina):
                print('\nPágina inválida')
                continue
        except ValueError:
            print('\nDebe ingresar un número')
            continue
        offset = (pagina - 1) * 50
        params = {
            'term': 'restaurants',
            'location': ciudad,
            'limit': 50,
            'offset': offset
        }
        response = requests.get(
            api_url,
            params=params,
            headers=headers
        )
        data = response.json()
        if 'businesses' not in data:
            print('\nLa API no pudo devolver '
                'resultados para esa página.')
            if 'error' in data:
                print('\nDetalle del error:')
                print(data['error'])
            continue
        negocios = data.get('businesses',[])
        nombres = []
        categorias = []
        ratings = []
        reviews = []
        precios = []
        for negocio in negocios:
            nombres.append(negocio.get('name'))
            categorias.append(
                ', '.join(
                    [
                        categoria['title']
                        for categoria
                        in negocio['categories']
                    ]
                )
            )
            ratings.append(
                negocio.get('rating')
            )
            reviews.append(
                negocio.get('review_count')
            )
            precio = negocio.get(
                'price',
                'No especificado'
            )
            #Correcion de datos analizados incorrectamente para facilidad del usuario
            #Creacion y uso del DataFrame
            if precio == '$$':
                precio = 'Medio'
            elif precio == '$$$':
                precio = 'Alto'
            elif precio == '$$$$':
                precio = 'Muy Alto'
            else:
                precio = 'No especificado'
            precios.append(precio)
        yelp = pd.DataFrame({
            'Nombre': nombres,
            'Categoría': categorias,
            'Rating': ratings,
            'Reseñas': reviews,
            'Precio': precios
        })
        #Tercer ciclo while donde el usuario obtendra accesos a diferentes opciones para la obtencion y el analisis de datos y que al salir pueda volver al menu principal y no detenga el codigo por completo
        #El usuario puede detener el analisis de la ciudad actual para cambiar a otra, hacer los otros analisis que necesite o seguir explorando los 200 datos de diferentes restaurantes que hay por ciudad si asi lo desea!
        #Nuevamente se hace control de excepciones
        while True:
            print(f'\nCiudad seleccionada: {ciudad}')
            print(f'Página cargada: {pagina}')
            print('\n1. Ver restaurantes')
            print('2. Top 10 mejor calificados')
            print('3. Top 10 más reseñados')
            print('4. Categorías más frecuentes')
            print('5. Estadísticas generales')
            print('6. Cargar otra página')
            print('7. Volver al menú principal')
            opcion = input('\nSeleccione una opción: ')
            if opcion == '1':
                max_filas = len(yelp)
                while True:
                    try:
                        print(f'\nRegistros disponibles: '
                            f'0 a {max_filas}')
                        inicio = int(input('Fila inicial: '))
                        fin = int(
                            input('Fila final: '))
                        if not (0 <= inicio < fin <= max_filas):
                            print('\nRango inválido')
                            continue
                        display(yelp.iloc[inicio:fin])
                        while True:
                            accion = input(
                                '\n1. Ver otras filas\n'
                                '2. Volver al menú\n'
                                'Seleccione: '
                            )
                            if accion == '1':
                                break
                            elif accion == '2':
                                raise StopIteration
                            else:
                                print('\nOpción inválida')
                    except ValueError:
                        print('\nDebe ingresar números')
                    except StopIteration:
                        break
            elif opcion == '2':
                top_rating = (yelp.sort_values(by='Rating',ascending=False).head(10).reset_index(drop=True))
                print('\nTop 10 mejor calificados\n')
                display(top_rating)
            elif opcion == '3':
                top_reviews = (yelp.sort_values(by='Reseñas',ascending=False).head(10).reset_index(drop=True))
                print('\nTop 10 más reseñados\n')
                display(top_reviews)
            elif opcion == '4':
                categorias_df = (yelp['Categoría'].str.split(', ').explode().value_counts().reset_index())
                categorias_df.columns = ['Categoría','Cantidad']
                print('\nCategorías más frecuentes '
                    'en los restaurantes cargados\n')
                display(categorias_df.head(10))
            elif opcion == '5':
                estadisticas = pd.DataFrame({
                    'Indicador': [
                        'Restaurantes analizados',
                        'Rating promedio',
                        'Rating máximo',
                        'Rating mínimo',
                        'Promedio de reseñas'
                    ],
                    'Valor': [
                        len(yelp),
                        round(np.mean(yelp['Rating']),2),np.max(yelp['Rating']),
                        np.min(yelp['Rating']),round(np.mean(yelp['Reseñas']),0)
                    ]
                })
                print('\nEstadísticas generales\n')
                display(
                    estadisticas
                )
                restaurante_popular = (yelp.loc[yelp['Reseñas'].idxmax()])
                popular = pd.DataFrame({
                    'Nombre': [restaurante_popular['Nombre']],
                    'Rating': [restaurante_popular['Rating']],
                    'Reseñas': [restaurante_popular['Reseñas']],
                    'Precio': [restaurante_popular['Precio']]
                })
                print('\nRestaurante más popular\n')
                display(popular)
            elif opcion == '6':
                break
            elif opcion == '7':
                volver_ciudades = True
                break
            else:
                print('\nOpción inválida')


Ciudades disponibles

1. Miami
2. Denver
3. Boston
4. San Diego
5. Dallas
6. NYC
7. Seattle
8. Chicago
9. Houston
10. Phoenix
0. Salir

Debe ingresar un número

Ciudades disponibles

1. Miami
2. Denver
3. Boston
4. San Diego
5. Dallas
6. NYC
7. Seattle
8. Chicago
9. Houston
10. Phoenix
0. Salir

Debe ingresar un número

Ciudades disponibles

1. Miami
2. Denver
3. Boston
4. San Diego
5. Dallas
6. NYC
7. Seattle
8. Chicago
9. Houston
10. Phoenix
0. Salir

Debe ingresar un número

Ciudades disponibles

1. Miami
2. Denver
3. Boston
4. San Diego
5. Dallas
6. NYC
7. Seattle
8. Chicago
9. Houston
10. Phoenix
0. Salir

Ciudad inválida

Ciudades disponibles

1. Miami
2. Denver
3. Boston
4. San Diego
5. Dallas
6. NYC
7. Seattle
8. Chicago
9. Houston
10. Phoenix
0. Salir

Debe ingresar un número

Ciudades disponibles

1. Miami
2. Denver
3. Boston
4. San Diego
5. Dallas
6. NYC
7. Seattle
8. Chicago
9. Houston
10. Phoenix
0. Salir

Consultando Yelp para Miami...

Yelp registra aproximadamente 5,80

,Nombre,Categoría,Rating,Reseñas,Precio
0,Versailles,"Cuban, Coffee & Tea, Sandwiches",4.0,7031,Medio
1,Old's Havana Cuban Bar & Cocina,"Cuban, Bars, Venues & Event Spaces",4.4,3220,Medio
2,Crazy About You,"Mediterranean, Italian, Spanish",4.2,2895,Alto
3,Kyu,Asian Fusion,4.4,2117,Alto
4,Fratellino,Italian,4.8,1915,Alto
5,Lung Yai Thai Tapas,"Tapas/Small Plates, Thai",4.0,1814,Medio
6,Bombay Darbar,Indian,4.5,1713,Medio
7,Glass & Vine,"New American, Breakfast & Brunch",4.1,1711,Medio
8,Sanguich,"Cuban, Sandwiches",4.5,1501,Medio
9,Apocalypse BBQ,"Food Trucks, Barbeque, Empanadas",4.4,1180,Medio



Ciudad seleccionada: Miami
Página cargada: 1

1. Ver restaurantes
2. Top 10 mejor calificados
3. Top 10 más reseñados
4. Categorías más frecuentes
5. Estadísticas generales
6. Cargar otra página
7. Volver al menú principal

La API no pudo devolver resultados para esa página.

Detalle del error:
{'code': 'VALIDATION_ERROR', 'description': 'Too many results requested, limit+offset must be <= 240.'}

Ciudad seleccionada: Miami
Página cargada: 1

1. Ver restaurantes
2. Top 10 mejor calificados
3. Top 10 más reseñados
4. Categorías más frecuentes
5. Estadísticas generales
6. Cargar otra página
7. Volver al menú principal

Registros disponibles: 0 a 50

Rango inválido

Registros disponibles: 0 a 50


,Nombre,Categoría,Rating,Reseñas,Precio
0,Fratellino,Italian,4.8,1915,Alto
1,Old's Havana Cuban Bar & Cocina,"Cuban, Bars, Venues & Event Spaces",4.4,3220,Medio
2,Salty Flame,"Asian Fusion, Steakhouses, Cocktail Bars",4.3,272,No especificado
3,Crazy About You,"Mediterranean, Italian, Spanish",4.2,2895,Alto
4,House of Food Porn,"Sushi Bars, Asian Fusion, Wine Bars",4.8,177,No especificado
5,Bayshore Club Bar & Grill,"Seafood, American, Wine Bars",4.3,1057,Alto
6,Giselle Miami,"Mediterranean, Lounges, Asian Fusion",4.6,507,No especificado
7,Versailles,"Cuban, Coffee & Tea, Sandwiches",4.0,7031,Medio
8,Chef Allen's Farm to Table,"Vegetarian, Food Stands",4.6,14,Alto
9,Capriccio,"Italian, Pizza, Steakhouses",4.5,175,No especificado



Registros disponibles: 0 a 50


...


Ciudad seleccionada: Denver
Página cargada: 2

1. Ver restaurantes
2. Top 10 mejor calificados
3. Top 10 más reseñados
4. Categorías más frecuentes
5. Estadísticas generales
6. Cargar otra página
7. Volver al menú principal

Ciudades disponibles

1. Miami
2. Denver
3. Boston
4. San Diego
5. Dallas
6. NYC
7. Seattle
8. Chicago
9. Houston
10. Phoenix
0. Salir

Consultando Yelp para Phoenix...

Yelp registra aproximadamente 3,000 restaurantes en Phoenix.

Por limitaciones de la API de Yelp solo pueden consultarse los primeros 1000 resultados.

Restaurantes disponibles para análisis: 1,000
Puede seleccionar páginas entre 1 y 20.

Ciudad seleccionada: Phoenix
Página cargada: 1

1. Ver restaurantes
2. Top 10 mejor calificados
3. Top 10 más reseñados
4. Categorías más frecuentes
5. Estadísticas generales
6. Cargar otra página
7. Volver al menú principal

Top 10 mejor calificados



,Nombre,Categoría,Rating,Reseñas,Precio
0,Pretty Penny,"Cocktail Bars, Seafood, Steakhouses",4.9,90,No especificado
1,Vecina,"Latin American, New American",4.7,446,Medio
2,Orchard Tavern,"Gastropubs, Bars",4.7,164,No especificado
3,Let’s Toast,"Korean, Food Trucks, Sandwiches",4.6,42,No especificado
4,Industry Standard,"New American, Cocktail Bars",4.6,103,Medio
5,Urban Fraiche Restaurant,Mediterranean,4.6,120,No especificado
6,Glai Baan,Thai,4.6,1547,Medio
7,théa,"Mediterranean, Cocktail Bars, Desserts",4.6,635,Alto
8,Bobby Q,"Barbeque, New American",4.5,5368,Medio
9,Tesota,"Comfort Food, Bars, Pizza",4.5,302,No especificado



Ciudad seleccionada: Phoenix
Página cargada: 1

1. Ver restaurantes
2. Top 10 mejor calificados
3. Top 10 más reseñados
4. Categorías más frecuentes
5. Estadísticas generales
6. Cargar otra página
7. Volver al menú principal

Top 10 más reseñados



,Nombre,Categoría,Rating,Reseñas,Precio
0,Bobby Q,"Barbeque, New American",4.5,5368,Medio
1,The Arrogant Butcher,"American, Seafood, Cocktail Bars",4.0,3626,Medio
2,Cibo,"Pizza, Italian, Sandwiches",4.4,3084,Medio
3,Culinary Dropout,New American,4.1,2997,Medio
4,The Henry,"New American, Breakfast & Brunch, Coffee & Tea",4.2,2575,Medio
5,Chelsea's Kitchen,New American,4.3,2414,Medio
6,Buck & Rider,"Seafood, Breakfast & Brunch, Live/Raw Food",4.4,2370,Alto
7,The Clever Koi,"Asian Fusion, Ramen, Cocktail Bars",4.2,2201,Alto
8,The Parlor,"Pizza, Italian, Salad",4.4,2128,Medio
9,Windsor,"New American, Breakfast & Brunch, Cocktail Bars",4.3,2022,Medio



Ciudad seleccionada: Phoenix
Página cargada: 1

1. Ver restaurantes
2. Top 10 mejor calificados
3. Top 10 más reseñados
4. Categorías más frecuentes
5. Estadísticas generales
6. Cargar otra página
7. Volver al menú principal

Ciudades disponibles

1. Miami
2. Denver
3. Boston
4. San Diego
5. Dallas
6. NYC
7. Seattle
8. Chicago
9. Houston
10. Phoenix
0. Salir

Hasta luego
